In [1]:
import json
import sys
from pathlib import Path

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

/Users/akshattewari/Projects/mlpurdue/Wall-E/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1027 18:04:07.944000 57756 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [10]:
# Resolve paths from either the repository root or the model/ folder.
repo_root = Path.cwd()
if not (repo_root / "keys" / "keys.json").exists() and repo_root.name == "model":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from model.bt_validator import BTValidationError, BehaviorTreeValidator

keys_path = repo_root / "keys" / "keys.json"
with keys_path.open("r", encoding="utf-8") as f:
    keys = json.load(f)
from dotenv import dotenv_values
config = dotenv_values("../../.env")
# print(config)
login(token=config["HF_TOKEN"])
model_id = "AIRLab-POLIMI/llama-3.2-1b-it-ft-lora-bt"
output_tree_path = repo_root / "model" / "tree.xml"
vocabulary_path = "./wall_e.yaml"
validator = BehaviorTreeValidator.from_yaml(vocabulary_path)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Use bf16 when the active CUDA device supports it; otherwise fall back to fp16.
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
else:
    compute_dtype = torch.float16

# 4-bit quantization keeps inference lightweight for the released 1B model.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

In [4]:
# The HF token from keys/keys.json is used above for authentication.
# No interactive Hugging Face login is required when HF_TOKEN is set.

In [5]:
# Load the fine-tuned BTGenBot-2 model from the Hugging Face repo in HF_LLAMA_FT_MODEL.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="mps",
)

Loading weights: 100%|██████████| 147/147 [00:02<00:00, 70.40it/s] 


In [6]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    
)

# Keep generation bounded to one behavior tree while allowing the model to sample valid alternatives.
generation_args = {
    "max_new_tokens": 1500,
    "return_full_text": False,
    "do_sample": True,
}

In [7]:
system_content = """
You are a helpful assistant that can assist with creating behavior trees.
Your task:
- Convert the provided summary of a behavior into an XML-formatted behavior tree.
- Ensure the behavior tree matches the description in the summary.
- The behavior tree must be compatible with the BehaviorTree.CPP library.
- Only use the actions and parameters provided in the action list below the summary.

Output Requirements:
- Output only the XML representation of the behavior tree. Do not include explanations, comments, or any additional text.
- Ensure all actions and parameters strictly match the provided list.
- If possible limit the use of SubTrees.

Please generate the behavior tree based on the summary and action list provided.
"""

In [8]:
user_content = """
This behavior tree manages a Wall-E like Robot. Create a tree for the robot to \"Find the ball\"

Control nodes: [Sequence, Fallback, Parallel (parameters: success_count, failure_count),
ReactiveSequence, ReactiveFallback, RoundRobin, PipelineSequence, SequenceStar,
RecoveryNode (parameters: number_of_retries)]

Decorator nodes: [Inverter, ForceSuccess, ForceFailure, Repeat (parameters: num_cycles),
RetryUntilSuccessful (parameters: num_attempts), KeepRunningUntilFailure,
Delay (parameters: delay_msec), Timeout (parameters: msec), RateController (parameters: hz),
DistanceController (parameters: distance)]

Actions: [SearchFor (parameters: object, optional: hint),
NavigateTo (parameters: target),
Say (parameters: text, optional: emotion),
Express (parameters: emotion),
LookAt (parameters: target),
Grab (parameters: object),
MoveArm (parameters: pose),
Release (optional: target),
Crush (parameters: object)]

Conditions: [IsVisible (parameters: object),
IsAt (parameters: target)]
"""

messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": user_content},
]

In [9]:
print(f"Repository root: {repo_root}")
print(f"Allowed primitives: {vocabulary_path}")
print(f"Generated tree will be saved to: {output_tree_path}")

Repository root: /Users/akshattewari/Projects/mlpurdue/Wall-E/akshat-work
Allowed primitives: ./wall_e.yaml
Generated tree will be saved to: /Users/akshattewari/Projects/mlpurdue/Wall-E/akshat-work/model/tree.xml


In [12]:
# Reject invalid generations and sample again, as described by the inference-time validator.
max_validation_attempts = 3
validation_errors = []

for attempt in range(1, max_validation_attempts + 1):
    output = pipe(messages, **generation_args)
    generated_text = output[0]["generated_text"]
    print(f"Generation attempt {attempt}:\n{generated_text}")
    # print(generated_text)
    # with open(f"purewrite_{attempt}.xml", "w") as f:
    #     f.write(generated_text)
    try:
        validator.validate(generated_text)
    except BTValidationError as error:
        validation_errors.append(f"Attempt {attempt}: {error}")
        print(f"Rejected: {error}")
        continue

    final_tree = generated_text.strip()
    print(f"Accepted behavior tree on attempt {attempt}.")
    break
else:
    details = "\n".join(validation_errors)
    raise BTValidationError(
        f"No valid behavior tree generated after {max_validation_attempts} attempts:\n{details}"
    )

print(final_tree)

Generation attempt 1:
<root main_tree_to_execute="WallE">
  <BehaviorTree ID="WallE">
    <DecoratorNode ID="RecoveryNode">
      <Fallback name="RecoveryFallback">
        <Sequence name="FindTheBall">
          <Inverter>
            <Condition ID="IsVisible" name="IsBallVisible"/>
          </Inverter>
          <Action ID="SearchFor" object="ball" optional="hint"/>
          <ForceSuccess>
            <Action ID="NavigateTo" target="ball"/>
          </ForceSuccess>
        </Sequence>
        <Parallel name="RecoveryParallel">
          <RetryUntilSuccessful number_of_retries="3">
            <Action ID="RetryUntilSuccessful" name="RetrySearch" object="ball"/>
          </RetryUntilSuccessful>
          <Action ID="Delay" delay_msec="1000"/>
        </Parallel>
      </Fallback>
    </DecoratorNode>
  </BehaviorTree>
</root>
Rejected: <root> must declare BTCPP_format="4".
Generation attempt 2:
<root BTCPP_format="4">
    <BehaviorTree ID="BehaviorTree">
        <Fallback name="Fin

BTValidationError: No valid behavior tree generated after 3 attempts:
Attempt 1: <root> must declare BTCPP_format="4".
Attempt 2: <root> must contain exactly one <BehaviorTree> element.
Attempt 3: Malformed XML: mismatched tag: line 85, column 18.

In [10]:
# Save the generated behavior tree for simulator or BehaviorTree.CPP use.
output_tree_path.parent.mkdir(parents=True, exist_ok=True)
with output_tree_path.open("w", encoding="utf-8") as f:
    f.write(final_tree)